In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
#  Spark Session
spark = SparkSession.builder \
    .appName("FraudDetectionSystem") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [3]:
#  Read from Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "transactions") \
    .load()

# Convert binary → string
df = df.selectExpr("CAST(value AS STRING)")

In [4]:
# Schema
schema = StructType() \
    .add("transaction_id", StringType()) \
    .add("user_id", IntegerType()) \
    .add("amount", IntegerType()) \
    .add("timestamp", IntegerType()) \
    .add("country", StringType()) \
    .add("city", StringType()) \
    .add("ip_address", StringType()) \
    .add("is_international", IntegerType()) \
    .add("device_type", StringType()) \
    .add("device_id", StringType()) \
    .add("is_new_device", IntegerType()) \
    .add("merchant", StringType()) \
    .add("category", StringType()) \
    .add("payment_method", StringType()) \
    .add("previous_fraud", IntegerType()) \
    .add("account_age_days", IntegerType()) \
    .add("transactions_last_1min", IntegerType())

In [5]:
# Parse JSON
df = df.select(from_json(col("value"), schema).alias("data")).select("data.*")

# FRAUD DETECTION (LOGIC)

In [6]:
df = df.withColumn("risk_score",
    (when(col("previous_fraud") == 1, 2).otherwise(0)) +
    (when(col("is_new_device") == 1, 1).otherwise(0)) +
    (when(col("amount") > 20000, 1).otherwise(0)) +
    (when(col("is_international") == 1, 1).otherwise(0)) +
    (when(col("transactions_last_1min") > 5, 1).otherwise(0)) +
    (when(col("account_age_days") < 7, 1).otherwise(0))
)

df = df.withColumn("risk_level",
    when(col("risk_score") >= 3, "HIGH")   
    .when(col("risk_score") >= 2, "MEDIUM")
    .otherwise("SAFE")
)

# 📊 ANALYTICS (REALISTIC)

In [7]:
# # 1. Risk Distribution
# fraud_summary = df.groupBy("risk_level").count()

# # 2️ Country-wise HIGH risk
# country_analysis = df.filter(col("risk_level") == "HIGH") \
#     .groupBy("country").count() \
#     .withColumnRenamed("count", "fraud_count")

# # 3️ Payment Analysis
# payment_analysis = df.groupBy("payment_method") \
#     .agg(
#         count("*").alias("total_txn"),
#         sum(when(col("risk_level") == "HIGH", 1).otherwise(0)).alias("fraud_txn")
#     )

# # 4️ Device Analysis
# device_analysis = df.groupBy("device_type") \
#     .agg(
#         count("*").alias("total_txn"),
#         sum(when(col("risk_level") == "HIGH", 1).otherwise(0)).alias("fraud_txn")
#     )

# # 5️ LIVE HIGH RISK TRANSACTIONS
# high_risk_txn = df.filter(col("risk_level") == "HIGH") \
#     .select("user_id", "amount", "country", "device_type", "risk_score", "risk_level")

fraud_summary = df.groupBy("risk_level").count()

all_txn = df.select(
    "user_id",
    "amount",
    "country",
    "device_type",
    "risk_score",
    "risk_level"
)

# OUTPUT STREAMS

In [8]:
# # Risk summary
# query1 = fraud_summary.writeStream \
#     .outputMode("complete") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# # Country fraud
# query2 = country_analysis.writeStream \
#     .outputMode("complete") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# # Payment analysis
# query3 = payment_analysis.writeStream \
#     .outputMode("complete") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# # Device analysis
# query4 = device_analysis.writeStream \
#     .outputMode("complete") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# # High risk live
# query5 = high_risk_txn.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# spark.streams.awaitAnyTermination()


query1 = fraud_summary.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

query2 = all_txn.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()



In [11]:
hdfs_query = df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "hdfs://namenode:8020/fraud_data") \
    .option("checkpointLocation", "/tmp/fraud_checkpoint_v2") \
    .start()
print("running")

running


In [ ]:
spark.streams.awaitAnyTermination()